<a href="https://colab.research.google.com/github/natacherchum/Descriptive_Statistics/blob/main/descriptive_stat_scipy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
url = "https://raw.githubusercontent.com/natacherchum/Descriptive_Statistics/refs/heads/main/descriptive_statistics_example1.csv"

values = np.genfromtxt(
    url,
    delimiter=",",
    skip_header=1,
    dtype=float,
    filling_values=np.nan,
)

print("Object type:", type(values))
print("Shape:", values.shape)
print("Number of dimensions:", values.ndim)
print("Data type:", values.dtype)

Object type: <class 'numpy.ndarray'>
Shape: (20, 3)
Number of dimensions: 2
Data type: float64


### Conclusion

All three variables are stored in a single two-dimensional NumPy array.

- The array contains **20 rows and 3 columns**.
- Rows represent participants.
- Columns represent height, weight, and daily physical activity.
- Every element has the same data type: **float64**.

Here, homogeneous means that all elements share the same data type,
even though the variables have different units of measurement.

In [ ]:
variable_names = ["height_cm", "weight_kg", "activity_minutes"]

mean_comparison = pd.DataFrame(
    {
        "Missing values": np.isnan(values).sum(axis=0),

        "Mean: propagate": stats.tmean(
            values,
            axis=0,
            nan_policy="propagate",
        ),

        "Mean: omit": stats.tmean(
            values,
            axis=0,
            nan_policy="omit",
        ),
    },
    index=variable_names,
)

display(mean_comparison)

,Missing values,Mean: propagate,Mean: omit
height_cm,0,170.91,170.910000
weight_kg,1,NaN,71.900000
activity_minutes,1,NaN,43.684211


### Conclusion

With `nan_policy="propagate"`, the mean is NaN for columns containing missing values.
With `nan_policy="omit"`, the mean uses only available observations.

Both methods give the same result for height because it has no missing values.

In [ ]:
variable_names = ["height_cm", "weight_kg", "activity_minutes"]

description = stats.describe(
    values,
    axis=0,
    ddof=1,
    bias=True,
    nan_policy="omit",
)

summary = pd.DataFrame(
    {
        "Observations": description.nobs,
        "Minimum": description.minmax[0],
        "Maximum": description.minmax[1],
        "Mean": description.mean,
        "Sample variance": description.variance,
        "Skewness": description.skewness,
        "Excess kurtosis": description.kurtosis,
    },
    index=variable_names,
)

display(summary)

,Observations,Minimum,Maximum,Mean,Sample variance,Skewness,Excess kurtosis
height_cm,20,157.9,184.1,170.910000,53.893579,-0.017779,-0.858296
weight_kg,19,52.8,92.5,71.900000,131.876667,-0.037872,-0.991498
activity_minutes,19,15.0,120.0,43.684211,694.005848,1.468023,1.978899


### Conclusion

`stats.describe` summarizes each variable, ignoring missing values.

- `ddof=1` calculates sample variance using the denominator n − 1.
- `bias=True` calculates skewness and excess kurtosis without bias corrections.

Median and mode are not included in `stats.describe`.

In [ ]:
mode_result = stats.mode(
    values,
    axis=0,
    nan_policy="omit",
    keepdims=False,
)

comparison = pd.DataFrame(
    {
        "Median": np.nanmedian(values, axis=0),
        "Mode": mode_result.mode,
        "Mode frequency": mode_result.count,
    },
    index=variable_names,
)

display(comparison)

,Median,Mode,Mode frequency
height_cm,170.75,157.9,1.0
weight_kg,72.80,52.8,1.0
activity_minutes,40.00,30.0,5.0


### Conclusion

Median and mode are not included in `stats.describe`.

`np.nanmedian` calculates the median, while `stats.mode` returns
a modal value and its frequency. Missing values are ignored.

Height and weight have no repeated values, so no value is more
frequent than the others. SciPy still returns one value with a count of 1.

For physical activity, the mode is 30 minutes, occurring 6 times.

In [ ]:
quantiles_summary = pd.DataFrame(
    np.nanquantile(
        values,
        [0.25, 0.50, 0.75],
        axis=0,
        method="linear",
    ),
    index=["Q1 (25%)", "Median (50%)", "Q3 (75%)"],
    columns=variable_names,
)

display(quantiles_summary)

,height_cm,weight_kg,activity_minutes
Q1 (25%),166.175,63.3,30.0
Median (50%),170.750,72.8,40.0
Q3 (75%),175.800,79.9,52.5


### Conclusion
`method="linear"` estimates a quantile by linear interpolation
between adjacent sorted values when its position falls between them.